# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library, utilizing Croissant schema `@id` identifiers for data access.

### Dataset Source
- Croissant Schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset high-level description
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")


## 2. Data Overview
List the available record sets and their fields using Croissant `@id` references. We use `dataset.record_sets` to inspect the data structure.

In [ ]:
# List all record sets (using @id)
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']} (name: {rs['name']})")

# For each record set, list fields (using @id)
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs['@id']} (name: {rs['name']})")
    if 'fields' in rs:
        for f in rs['fields']:
            fname = f.get('name', f['@id'])
            print(f"  - Field @id: {f['@id']} (name: {fname}, type: {f.get('dataType', 'N/A')})")
    else:
        print("  No fields detected.")

## 3. Data Extraction
Load records from a chosen record set into a DataFrame. All references use the Croissant `@id`.

In [ ]:
# Select the record set with tabular clinical records (replace with the actual @id if different)
# For this dataset, we'll extract the first tabular record set found.
tabular_rs = None
for rs in record_sets:
    if rs.get('fields'):
        tabular_rs = rs
        break
if tabular_rs is None:
    raise ValueError('No tabular record set found.')
record_set_id = tabular_rs['@id']
print(f"Selected record set @id: {record_set_id} (name: {tabular_rs['name']})")

# Load all record sets into dataframes using @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            print(f"Loaded {len(df)} records from {rs_id}")
        dataframes[rs_id] = df
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Show columns of main tabular dataset
main_df = dataframes[record_set_id]
print(f"\nFields in {record_set_id}:")
print(main_df.columns.tolist())

main_df.head()

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field using its Croissant `@id`, filter records above a threshold, normalize, and group by a categorical field.

In [ ]:
# Identify a numeric field @id (e.g., age at first diagnosis)
numeric_field_id = None
group_field_id = None
for field in tabular_rs.get('fields', []):
    if field.get('dataType', '').lower() in ['integer', 'float', 'number'] and not numeric_field_id:
        numeric_field_id = field['@id']
    if field.get('dataType', '').lower() == 'text' and not group_field_id:
        group_field_id = field['@id']
print(f"Numeric field @id: {numeric_field_id}")
print(f"Group-by field @id: {group_field_id}")

# Proceed if numeric field detected
if numeric_field_id is not None and numeric_field_id in main_df.columns:
    # Choose a filter threshold (e.g., 40 if 'Age')
    threshold = 40
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df[[numeric_field_id]].head())
    
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Optionally, group by a text field if available
    if group_field_id and group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
        print(f"\nGrouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No suitable numeric field found in selected record set.')

## 5. Visualization
We'll visualize the distribution of the selected numeric field and the group-averaged values, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_id is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped barplot if grouping field available
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.barplot(
            x=group_field_id,
            y=numeric_field_id,
            data=main_df[[group_field_id, numeric_field_id]].dropna(),
            estimator='mean',
            ci=None
        )
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=40)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- This notebook demonstrated the complete workflow for exploring and processing a Croissant-based FAIR² clinical dataset using the `mlcroissant` library.
- All entities (record sets, fields) were referenced using their Croissant `@id` for reproducibility.
- Example EDA and visualizations of demographic/numeric variables can be replicated for other fields using their respective `@id`.
